<div style="text-align: center;">
  <img src="./imagens/logo_novaims.png" alt="Logo" style="width: 150px; height: auto; margin-bottom: 10px;">
  <h1 style="margin: 0;"><strong>Machine Learning Project: Amazing International Airlines Inc.</strong></h1>
  <h2 style="margin: 0;"><strong>Part 1/2: Exploratory Data Analysis</strong></h2>
</div>

<div style="text-align: left; margin-top: 15px;">
  <p style="margin: 0;"><strong>Group 51:</strong></p>
  <ul style="margin: 0; padding-left: 20px;">
    <li>André Ferreira | 20250398</li>
    <li>Fausto Gomes | 20221915</li>
    <li>Maria Francisca Gonçalves | 20221942</li>
    <li>Miguel Matos | 20221925</li>
  </ul>
</div>

---
#### <font> Table of Contents </font> <a class="anchor" id='toc'></a> 
0. [Context](#introduction)
1. [Imports](#Imports)  
2. [Exploratory Data Analysis - Customer Data](#exploratory-data-analysis)

- 2.1. [Data Understanding](#21-data-understanding)
  - 2.1.1.[Descriptive Analysis](#descriptive-analysis)
  - 2.1.2.[Visualizations](#visualizations)

- 2.2. [GeoData](#geodata)

- 2.3. [Data Cleaning](#data-cleaning)



----

# <span style="color:#0097b2">0. Context</span>
[Back to TOC](#toc)

# <span style="color:#0097b2">1. Imports</span>
[Back to TOC](#toc)

In [1]:
from utils.functions import *
from utils.CustomPipeline import CustomPipeline
from pipelines.Missing_values_pipeline import MissingValuesDealer
from pipelines.Outliers_pipeline import OutliersDealer
from pipelines.Encoding_pipeline import EncodingDealer
from pipelines.Scaling_pipeline import ScalingDealer
from pipelines.Dimesionality_reduction_pipeline import Dimensionality_Reductor
from pipelines.Feature_selection_pipeline import Feature_Selector
import warnings

# Suppress this specific warning
warnings.filterwarnings('ignore', 
                       message='X does not have valid feature names',
                       category=UserWarning,
                       module='sklearn.utils.validation')



customer_data = pd.read_csv("../data/cleaned_data/customers_data_cleaned.csv", index_col= 0)
flights_data = pd.read_csv("../data/cleaned_data/flights_data_cleaned.csv", index_col= 0)

pd.set_option("display.max_columns", None)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.merge(
    customer_data, 
    flights_data, 
    on='Loyalty#', 
    how='outer' # To keep all clients registered
)

# The merge might create NaNs for customers in df_clean who had zero flight history
# For clustering, we should fill these with 0
features_to_fill = [
    'TotalFlights', 
    'TotalDistance', 
    'TotalPointsAccumulated', 
    'TotalPointsRedeemed'
]
data[features_to_fill] = data[features_to_fill].fillna(0)

# If 'RecencyInMonths' is NaN (from the 'left' merge), it means they never flew. 
# We can keep the 999 we set earlier, or set it again.
data['RecencyInMonths'] = data['RecencyInMonths'].fillna(999)

In [3]:
flights_data

,Loyalty#,TotalFlights,TotalDistance,TotalPointsAccumulated,TotalPointsRedeemed,LastFlightDate,RecencyInMonths,Pct_Spend_Month_1,Pct_Spend_Month_2,Pct_Spend_Month_3,Pct_Spend_Month_4,Pct_Spend_Month_5,Pct_Spend_Month_6,Pct_Spend_Month_7,Pct_Spend_Month_8,Pct_Spend_Month_9,Pct_Spend_Month_10,Pct_Spend_Month_11,Pct_Spend_Month_12,Flights_per_day
0,100018,229.9,530230.0,53014.30,20562.8,2021-12-01,1.018397,0.048421,0.060858,0.074292,0.061981,0.000000,0.132689,0.137946,0.108319,0.073489,0.065397,0.124303,0.112304,0.209954
1,100102,247.7,339114.6,33903.96,18760.6,2021-12-01,1.018397,0.060884,0.028640,0.033403,0.049635,0.046809,0.080404,0.114205,0.014635,0.179591,0.152183,0.057872,0.181741,0.226210
2,100140,216.8,432030.8,43192.58,4896.0,2021-11-01,2.003942,0.043443,0.000000,0.172908,0.012598,0.164527,0.065297,0.164342,0.008060,0.079373,0.009099,0.135146,0.145208,0.197991
3,100214,112.3,364601.7,36453.77,12908.6,2021-12-01,1.018397,0.000000,0.065188,0.057415,0.000000,0.000000,0.169642,0.000000,0.224220,0.135728,0.089849,0.007416,0.250542,0.102557
4,100272,186.4,429630.5,42953.25,10891.4,2021-11-01,2.003942,0.034945,0.110526,0.078136,0.041127,0.054967,0.167454,0.070783,0.179004,0.024562,0.044607,0.051625,0.142265,0.170228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16732,999902,267.1,610159.5,61006.55,10501.8,2021-10-01,3.022339,0.099376,0.089763,0.087597,0.051506,0.073032,0.148470,0.099877,0.122778,0.007860,0.081661,0.031620,0.106458,0.243927
16733,999911,0.0,0.0,0.00,0.0,NaN,999.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
16734,999940,85.5,238578.9,23855.59,5620.0,2021-12-01,1.018397,0.000000,0.000000,0.114145,0.000000,0.000000,0.000000,0.000000,0.121733,0.254003,0.000000,0.308023,0.202095,0.078082
16735,999982,22.0,52654.0,5264.00,0.0,2021-11-01,2.003942,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.017857,0.524696,0.000000,0.240502,0.216945,0.020091


In [4]:
data.head(10)

,Loyalty#,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType,rejoined_program,Days_in_prog,Cancelled_program,Enrollment_year,Enrollment_month,TotalFlights,TotalDistance,TotalPointsAccumulated,TotalPointsRedeemed,LastFlightDate,RecencyInMonths,Pct_Spend_Month_1,Pct_Spend_Month_2,Pct_Spend_Month_3,Pct_Spend_Month_4,Pct_Spend_Month_5,Pct_Spend_Month_6,Pct_Spend_Month_7,Pct_Spend_Month_8,Pct_Spend_Month_9,Pct_Spend_Month_10,Pct_Spend_Month_11,Pct_Spend_Month_12,Flights_per_day
0,100011,Ontario,Toronto,43.593187,-79.444335,W9D 4Q9,female,Bachelor,Suburban,NaN,Married,Star,2017-05-01,2017-05-01,NaN,Standard,0,0.0,0,2017,5,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100012,Quebec,Quebec City,46.759733,-71.141009,Y0C 7D6,male,Bachelor,Suburban,NaN,Single,Star,2019-02-27,2019-02-27,NaN,Standard,0,0.0,0,2019,2,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100013,Alberta,Edmonton,53.524829,-113.546357,L3S 9Y3,female,Bachelor,Rural,NaN,Married,Star,2017-09-20,2017-09-20,NaN,Standard,0,0.0,0,2017,9,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100014,New Brunswick,Moncton,46.051866,-64.825428,G2S 2B6,male,Bachelor,Rural,NaN,Married,Star,2020-11-28,2020-11-28,NaN,Standard,0,0.0,0,2020,11,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100015,Quebec,Quebec City,46.862970,-71.133444,B1Z 8T3,female,College,Urban,NaN,Married,Star,2020-04-09,2020-04-09,NaN,Standard,0,0.0,0,2020,4,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,100016,British Columbia,Dawson Creek,55.720562,-120.160090,M4A 1E4,female,Master,Suburban,NaN,Single,Star,2020-07-21,2020-07-21,NaN,Standard,0,0.0,0,2020,7,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,100017,British Columbia,Dawson Creek,55.751178,-120.264920,E0K 5I2,male,Master,Urban,NaN,Married,Star,2017-04-11,2017-04-11,NaN,Standard,0,0.0,0,2017,4,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,100018,Alberta,Edmonton,53.544388,-113.490930,T9G 1W3,female,Bachelor,Rural,82877.0,Married,Aurora,2019-08-09,NaN,7919.20,Standard,0,874.0,0,2019,8,229.9,530230.0,53014.30,20562.8,2021-12-01,1.018397,0.048421,0.060858,0.074292,0.061981,0.000000,0.132689,0.137946,0.108319,0.073489,0.065397,0.124303,0.112304,0.209954
8,100102,Ontario,Toronto,43.653225,-79.383186,M1R 4K3,male,College,Urban,0.0,Single,Nova,2016-03-09,NaN,2887.74,Standard,0,2122.0,0,2016,3,247.7,339114.6,33903.96,18760.6,2021-12-01,1.018397,0.060884,0.028640,0.033403,0.049635,0.046809,0.080404,0.114205,0.014635,0.179591,0.152183,0.057872,0.181741,0.226210
9,100140,British Columbia,Dawson Creek,55.759628,-120.237660,U5I 4F1,female,College,Suburban,0.0,Divorced,Nova,2019-07-30,NaN,2838.07,Standard,0,884.0,0,2019,7,216.8,432030.8,43192.58,4896.0,2021-11-01,2.003942,0.043443,0.000000,0.172908,0.012598,0.164527,0.065297,0.164342,0.008060,0.079373,0.009099,0.135146,0.145208,0.197991


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16757 entries, 0 to 16756
Data columns (total 40 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Loyalty#                 16757 non-null  int64  
 1   Province or State        16757 non-null  object 
 2   City                     16757 non-null  object 
 3   Latitude                 16757 non-null  float64
 4   Longitude                16757 non-null  float64
 5   Postal code              16757 non-null  object 
 6   Gender                   16757 non-null  object 
 7   Education                16757 non-null  object 
 8   Location Code            16757 non-null  object 
 9   Income                   16737 non-null  float64
 10  Marital Status           16757 non-null  object 
 11  LoyaltyStatus            16757 non-null  object 
 12  EnrollmentDateOpening    16757 non-null  object 
 13  CancellationDate         2286 non-null   object 
 14  Customer Lifetime Valu

### Feature Engeneering

In [6]:
data["Avg_spent_per_flight"] = np.where(data["EnrollmentDateOpening"] >= "2019-01-01",
                                        data["Customer Lifetime Value"]/(data["Flights_per_day"] * data["Days_in_prog"]),
                                        np.nan)

"Inf" means infinity, which happens when the divisor is zero, so when "Days_in_prog" or "Flights_per_day" is zero. To handle this, we will feel it with Nan for now.

In [7]:
data["Avg_spent_per_flight"] = data["Avg_spent_per_flight"].replace(np.inf, np.nan)

In [8]:
data["Avg_spent_per_flight"].describe()

count      7491.000000
mean       1960.837518
std       12811.625099
min           6.359410
25%          51.737521
50%         147.920551
75%         571.234834
max      749883.717188
Name: Avg_spent_per_flight, dtype: float64

In [9]:
data = data.drop(columns = ["EnrollmentDateOpening", "CancellationDate", "Loyalty#"])

In [10]:
data.head(10)

,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,Customer Lifetime Value,EnrollmentType,rejoined_program,Days_in_prog,Cancelled_program,Enrollment_year,Enrollment_month,TotalFlights,TotalDistance,TotalPointsAccumulated,TotalPointsRedeemed,LastFlightDate,RecencyInMonths,Pct_Spend_Month_1,Pct_Spend_Month_2,Pct_Spend_Month_3,Pct_Spend_Month_4,Pct_Spend_Month_5,Pct_Spend_Month_6,Pct_Spend_Month_7,Pct_Spend_Month_8,Pct_Spend_Month_9,Pct_Spend_Month_10,Pct_Spend_Month_11,Pct_Spend_Month_12,Flights_per_day,Avg_spent_per_flight
0,Ontario,Toronto,43.593187,-79.444335,W9D 4Q9,female,Bachelor,Suburban,NaN,Married,Star,NaN,Standard,0,0.0,0,2017,5,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Quebec,Quebec City,46.759733,-71.141009,Y0C 7D6,male,Bachelor,Suburban,NaN,Single,Star,NaN,Standard,0,0.0,0,2019,2,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Alberta,Edmonton,53.524829,-113.546357,L3S 9Y3,female,Bachelor,Rural,NaN,Married,Star,NaN,Standard,0,0.0,0,2017,9,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,New Brunswick,Moncton,46.051866,-64.825428,G2S 2B6,male,Bachelor,Rural,NaN,Married,Star,NaN,Standard,0,0.0,0,2020,11,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Quebec,Quebec City,46.862970,-71.133444,B1Z 8T3,female,College,Urban,NaN,Married,Star,NaN,Standard,0,0.0,0,2020,4,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,British Columbia,Dawson Creek,55.720562,-120.160090,M4A 1E4,female,Master,Suburban,NaN,Single,Star,NaN,Standard,0,0.0,0,2020,7,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,British Columbia,Dawson Creek,55.751178,-120.264920,E0K 5I2,male,Master,Urban,NaN,Married,Star,NaN,Standard,0,0.0,0,2017,4,0.0,0.0,0.00,0.0,NaN,999.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Alberta,Edmonton,53.544388,-113.490930,T9G 1W3,female,Bachelor,Rural,82877.0,Married,Aurora,7919.20,Standard,0,874.0,0,2019,8,229.9,530230.0,53014.30,20562.8,2021-12-01,1.018397,0.048421,0.060858,0.074292,0.061981,0.000000,0.132689,0.137946,0.108319,0.073489,0.065397,0.124303,0.112304,0.209954,43.156382
8,Ontario,Toronto,43.653225,-79.383186,M1R 4K3,male,College,Urban,0.0,Single,Nova,2887.74,Standard,0,2122.0,0,2016,3,247.7,339114.6,33903.96,18760.6,2021-12-01,1.018397,0.060884,0.028640,0.033403,0.049635,0.046809,0.080404,0.114205,0.014635,0.179591,0.152183,0.057872,0.181741,0.226210,NaN
9,British Columbia,Dawson Creek,55.759628,-120.237660,U5I 4F1,female,College,Suburban,0.0,Divorced,Nova,2838.07,Standard,0,884.0,0,2019,7,216.8,432030.8,43192.58,4896.0,2021-11-01,2.003942,0.043443,0.000000,0.172908,0.012598,0.164527,0.065297,0.164342,0.008060,0.079373,0.009099,0.135146,0.145208,0.197991,16.215326


# Data Preproc Parameters

In [13]:
km = KMeans(n_clusters= 5)

pipeline = CustomPipeline(MissingValuesDealer(),
                          OutliersDealer(),
                          EncodingDealer(),
                          ScalingDealer(),
                          Dimensionality_Reductor(),
                          Feature_Selector(),
                          km)

params = {
            "imputer__imputation_method": ["knn", "simple", "iterative"],
            "imputer__knn_neighbors": np.arange(5,21),
            "imputer__knn_scaling_method" : ["standard", "minmax", "robust"],
            "outlier_remover__outlier_method" : ["Isolation_Forest", "LOF", "z_score"],
            "outlier_remover__threshold" : [2, 2.5, 3],
            "outlier_remover__contamination_IF" : [0.05, 0.15, 0.3, 0.5],
            "outlier_remover__n_neighbors": [30, 50, 70, 90, 100],
            "outlier_remover__contamination_LOF" : [0.05, 0.15, 0.3, 0.5],
            "encoder__method" : ["onehot"],
            'reducer__method': ['pca', 'kernel_pca', 'svd'],
            'reducer__n_components': np.arange(2, 20),
            'reducer__kernel': ['rbf', 'linear', 'poly', 'sigmoid', 'cosine']
            #'selector__method': ['variance', 'correlation', 'pca_loading'],
            #'selector__threshold': uniform(0.0, 0.3),
            #'selector__corr_threshold': uniform(0.7, 0.3), 
            #'selector__n_features': randint(3, 30)
    }

# Use a single train-test split where both are the full dataset
cv = [(np.arange(len(data)), np.arange(len(data)))]

search = RandomizedSearchCV(
    pipeline,
    params,
    n_iter=10,
    scoring= clustering_scorer,  
    cv= cv,
    verbose= 3,
    n_jobs = 4)

THE ISSUE IS WHEN THE SELECTOR IS NOT "VARIANCE"!!!!
OR Other thing!!!
See with Claude

In [14]:
search.fit(data)

Fitting 1 folds for each of 10 candidates, totalling 10 fits
[CV 1/1] END encoder__method=onehot, imputer__imputation_method=iterative, imputer__knn_neighbors=14, imputer__knn_scaling_method=robust, outlier_remover__contamination_IF=0.5, outlier_remover__contamination_LOF=0.5, outlier_remover__n_neighbors=50, outlier_remover__outlier_method=Isolation_Forest, outlier_remover__threshold=2.5, reducer__kernel=sigmoid, reducer__method=kernel_pca, reducer__n_components=19;, score=nan total time=   2.4s
[CV 1/1] END encoder__method=onehot, imputer__imputation_method=knn, imputer__knn_neighbors=16, imputer__knn_scaling_method=minmax, outlier_remover__contamination_IF=0.05, outlier_remover__contamination_LOF=0.05, outlier_remover__n_neighbors=100, outlier_remover__outlier_method=Isolation_Forest, outlier_remover__threshold=3, reducer__kernel=rbf, reducer__method=svd, reducer__n_components=13;, score=nan total time=   8.2s
[CV 1/1] END encoder__method=onehot, imputer__imputation_method=iterative

KeyboardInterrupt: 

In [ ]:
# Test the pipeline with fixed parameters
from sklearn.cluster import KMeans

test_pipeline = CustomPipeline(
    imputer=MissingValuesDealer(imputation_method='simple'),
    outlier_remover=OutliersDealer(outlier_method='z_score', threshold=3, z_columns=[]),
    encoder=EncodingDealer(method='onehot'),
    scaler=ScalingDealer(scaler_name='robust'),
    reducer=Dimensionality_Reductor(method='pca', n_components=5),
    selector=Feature_Selector(method='variance', threshold=0.01),
    model=KMeans(n_clusters=3, random_state=42)
)

# Try to fit
try:
    labels = test_pipeline.fit_predict(data)
    print(f"Labels: {labels[:10]}")
    print(f"Unique clusters: {len(set(labels))}")
    print(f"Transformed shape: {test_pipeline.X_.shape}")
    
    # Try to score
    from sklearn.metrics import silhouette_score
    score = silhouette_score(test_pipeline.X_, labels)
    print(f"Silhouette score: {score}")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Labels: [0 0 0 0 0 0 0 0 0 0]
Unique clusters: 3
Transformed shape: (16757, 5)
Silhouette score: 0.9736509515139131
